# Convolution: hands-on (optional)
**LLMs, Animated · Lesson P5.4**

This notebook is optional practice for the lesson. Each exercise has:
1. a cell with a `# TODO` for you to fill in,
2. a **check** cell with `assert`s. Run it; silence (or a ✓) means success.

Stuck? Solutions are at the very bottom. Try first, though: *struggling to recall is what makes it stick.*

Only `numpy` and `matplotlib` are needed (both pre-installed on Colab).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
print("ready")

## 1 · 1D convolution, deep-learning style
Implement `conv1d(x, w)` with **no padding and stride 1**:
$$y[i] = \sum_{j=0}^{k-1} w[j]\,x[i + j], \qquad i = 0, \dots, n - k$$
Use plain loops or slices (not `np.convolve`); you'll compare against NumPy in the check.

In [ ]:
def conv1d(x, w):
    x, w = np.asarray(x, dtype=float), np.asarray(w, dtype=float)
    # TODO
    raise NotImplementedError("Exercise 1")

**Check 1**

In [ ]:
x = [2, 2, 2, 6, 6, 6, 2, 2]
assert np.allclose(conv1d(x, [0.25, 0.5, 0.25]), [2, 3, 5, 6, 5, 3])      # blur (from the animation)
assert np.allclose(conv1d(x, [-1, 0, 1]), [0, 4, 4, 0, -4, -4])            # edge detector
r = np.random.default_rng(0)
xr, wr = r.standard_normal(50), r.standard_normal(5)
assert np.allclose(conv1d(xr, wr), np.correlate(xr, wr, "valid"))         # cross-correlation...
assert np.allclose(conv1d(xr, wr), np.convolve(xr, wr[::-1], "valid"))    # ...is convolution with a flipped kernel
print("✓ exercise 1:  np.convolve flips the kernel:", np.convolve(x, [-1, 0, 1], "valid"))

## 2 · The output-size formula
Implement `out_size(n, k, p, s)` $= \lfloor (n + 2p - k)/s \rfloor + 1$.

In [ ]:
def out_size(n, k, p=0, s=1):
    # TODO
    raise NotImplementedError("Exercise 2")

**Check 2**

In [ ]:
assert out_size(10, 3) == 8
assert out_size(224, 7, p=3, s=2) == 112       # ResNet stem
assert out_size(32, 3, p=1, s=1) == 32         # "same" padding
assert out_size(32, 3, p=1, s=2) == 16
assert out_size(32, 5) == 28                   # LeNet-5's first layer
assert out_size(8, 3, s=2) == 3                # the animation's stride step
print("✓ exercise 2")

## 3 · Padding and stride in 1D
Extend exercise 1: `conv1d_ps(x, w, pad=0, stride=1)` first adds `pad` zeros at **each** end, then keeps every `stride`-th window. Its length must match `out_size`.

In [ ]:
def conv1d_ps(x, w, pad=0, stride=1):
    # TODO: np.pad, then reuse conv1d and take every stride-th output
    raise NotImplementedError("Exercise 3")

**Check 3**

In [ ]:
assert np.allclose(conv1d_ps(x, [-1, 0, 1], pad=1), [2, 0, 4, 4, 0, -4, -4, -2])   # padding step of the animation
assert np.allclose(conv1d_ps(x, [-1, 0, 1], stride=2), [0, 4, -4])                  # stride step
for n, k, p, s in [(10, 3, 0, 1), (11, 4, 2, 3), (32, 5, 1, 2), (7, 7, 0, 1)]:
    assert len(conv1d_ps(np.ones(n), np.ones(k), pad=p, stride=s)) == out_size(n, k, p, s), (n, k, p, s)
print("✓ exercise 3")

## 4 · 2D convolution
Implement `conv2d(X, W)` for a 2D array `X` (H×W) and a square kernel `W` (k×k), no padding, stride 1:
$$Y[r, c] = \sum_{i}\sum_{j} W[i, j]\,X[r + i,\, c + j]$$

In [ ]:
def conv2d(X, W):
    X, W = np.asarray(X, dtype=float), np.asarray(W, dtype=float)
    # TODO: two loops over output positions, multiply the patch by W and sum
    raise NotImplementedError("Exercise 4")

**Check 4**

In [ ]:
img = np.where(np.arange(6) >= 3, 5.0, 1.0)[None, :].repeat(6, axis=0)   # the animation's image
K = np.array([[-1, 0, 1]] * 3, dtype=float)                               # vertical-edge kernel
assert np.allclose(conv2d(img, K), [[0, 12, 12, 0]] * 4)
shifted = np.where(np.arange(6) >= 4, 5.0, 1.0)[None, :].repeat(6, axis=0)
assert np.allclose(conv2d(shifted, K), [[0, 0, 12, 12]] * 4)              # translation equivariance
r = np.random.default_rng(1)
Xr, Wr = r.standard_normal((9, 7)), r.standard_normal((3, 3))
ref = np.einsum("rcij,ij->rc", np.lib.stride_tricks.sliding_window_view(Xr, (3, 3)), Wr)
assert np.allclose(conv2d(Xr, Wr), ref)
print("✓ exercise 4")

## 5 · Edge-detect a tiny image
Below is a 12×12 synthetic image: a bright square on a dark background. The Sobel kernels respond to horizontal and vertical brightness changes.

Fill in `edges`: the **gradient magnitude** $\sqrt{G_x^2 + G_y^2}$, where $G_x$ = `conv2d(image, SOBEL_X)` and $G_y$ = `conv2d(image, SOBEL_Y)`. It's a 10×10 array (why 10? check with `out_size`).

In [ ]:
image = np.zeros((12, 12))
image[3:9, 4:10] = 1.0          # a bright 6x6 square
SOBEL_X = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=float)
SOBEL_Y = SOBEL_X.T
plt.figure(figsize=(3, 3)); plt.imshow(image, cmap="gray"); plt.title("input"); plt.axis("off"); plt.show()

In [ ]:
edges = None   # TODO

**Check 5**

In [ ]:
assert edges is not None and edges.shape == (out_size(12, 3), out_size(12, 3)) == (10, 10)
assert np.allclose(edges[0], 0) and np.allclose(edges[-1], 0)       # far from the square: flat, no response
assert np.isclose(edges[4, 4], 0.0)                                   # inside the square: flat too
assert edges[4, 2] > 3 and edges[4, 8] > 3                            # left and right borders light up
assert edges[2, 5] > 3 and edges[7, 5] > 3                            # top and bottom borders
plt.figure(figsize=(3, 3)); plt.imshow(edges, cmap="magma"); plt.title("edge strength"); plt.axis("off"); plt.show()
print("✓ exercise 5:  max edge strength", edges.max())

## Experiment · kernels as filters
Try other kernels on `image`, or on the noisy version below: a 3×3 box blur (`np.ones((3, 3)) / 9`), a sharpen kernel (`[[0, -1, 0], [-1, 5, -1], [0, -1, 0]]`), or the Sobel kernels on the blurred image. Notice how blurring first makes the edges thicker but less noisy.

In [ ]:
noisy_img = image + 0.15 * np.random.default_rng(2).standard_normal(image.shape)
blur = np.ones((3, 3)) / 9
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, (title, im) in zip(axes, [("noisy", noisy_img), ("blurred", conv2d(noisy_img, blur)),
                                   ("edges of blurred", np.hypot(conv2d(conv2d(noisy_img, blur), SOBEL_X), conv2d(conv2d(noisy_img, blur), SOBEL_Y)))]):
    ax.imshow(im, cmap="gray"); ax.set_title(title); ax.axis("off")
plt.show()

## Recap
- Deep-learning "convolution" is cross-correlation: slide, multiply, add. `np.convolve` flips the kernel first.
- Output size $\lfloor (n + 2p - k)/s \rfloor + 1$; you checked it against real outputs.
- 2D convolution with a $[-1, 0, 1]$ or Sobel kernel finds edges; shifting the input shifts the output.

Back to the course and hit **Mark complete** to add the review cards.

---
# Solutions
Only peek after a real attempt.

### Exercise 1

In [ ]:
# Solution (copy into the exercise cell above)
def conv1d(x, w):
    x, w = np.asarray(x, dtype=float), np.asarray(w, dtype=float)
    n, k = len(x), len(w)
    return np.array([np.dot(w, x[i:i + k]) for i in range(n - k + 1)])

### Exercise 2

In [ ]:
# Solution (copy into the exercise cell above)
def out_size(n, k, p=0, s=1):
    return (n + 2 * p - k) // s + 1

### Exercise 3

In [ ]:
# Solution (copy into the exercise cell above)
def conv1d_ps(x, w, pad=0, stride=1):
    xp = np.pad(np.asarray(x, dtype=float), pad)
    return conv1d(xp, w)[::stride]

### Exercise 4

In [ ]:
# Solution (copy into the exercise cell above)
def conv2d(X, W):
    X, W = np.asarray(X, dtype=float), np.asarray(W, dtype=float)
    k = W.shape[0]
    H, Wd = X.shape[0] - k + 1, X.shape[1] - k + 1
    Y = np.zeros((H, Wd))
    for r in range(H):
        for c in range(Wd):
            Y[r, c] = np.sum(W * X[r:r + k, c:c + k])
    return Y

### Exercise 5

In [ ]:
# Solution (copy into the exercise cell above)
gx, gy = conv2d(image, SOBEL_X), conv2d(image, SOBEL_Y)
edges = np.sqrt(gx ** 2 + gy ** 2)